In [18]:
# Recharge automatiquement les modules du projet quand leur code change.
# ATTENTION : après BEAUCOUP de modifs de env/, un simple reload ne suffit pas
# (les imports croisés gardent des constantes périmées). Le garde-fou en bas
# force alors un "Kernel > Restart Kernel".
%load_ext autoreload
%autoreload 2

import importlib
import json
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../env"))
sys.path.append(os.path.abspath("../pricing_agent"))

# Recharge la chaîne complète DANS L'ORDRE des dépendances (noms en clair -> pas
# de collision avec une variable locale) :
#   refdata (constantes) -> agents (acheteur MNL) -> market (simulateur) -> gym_wrapper (env RL)
def _reload_project():
    for _name in ("refdata", "agents", "market", "gym_wrapper"):
        importlib.reload(importlib.import_module(_name))

_reload_project()
from gym_wrapper import STATE_BINS, TicketEnv, discretize_observation

HORIZON = 120
env = TicketEnv(horizon=HORIZON)          # randomisation de domaine active par défaut

# Garde-fou : si l'env n'a pas les champs récents, le reload a échoué -> redémarrer le kernel.
_info0 = env.reset(seed=0)[1]
if "ca_total" not in _info0 or not hasattr(env, "steps_per_decision"):
    raise RuntimeError("Environnement périmé — fais 'Kernel > Restart Kernel' puis Run All.")

print("Environnement chargé.")
print("  observation :", env.observation_space, "(temps, inventaire, popularité, rythme, prix/réf, FCI, dernier prix)")
print("  actions     :", env.action_space, "->", env.num_price_levels, "paliers de prix")
print("  cadence     : 1 révision de prix tous les", env.steps_per_decision,
      "pas ->", HORIZON // env.steps_per_decision, "décisions / épisode")
print("  grille Q    :", STATE_BINS, "->", int(np.prod(STATE_BINS)), "états (6 premières dims de l'observation)")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Environnement chargé.
  observation : Box(0.0, 1.0, (7,), float32) (temps, inventaire, popularité, rythme, prix/réf, FCI, dernier prix)
  actions     : Discrete(10) -> 10 paliers de prix
  cadence     : 1 révision de prix tous les 10 pas -> 12 décisions / épisode
  grille Q    : (6, 6, 5, 4, 3, 3) -> 6480 états (6 premières dims de l'observation)


In [ ]:
# --- Agent tabulaire par CONTRÔLE MONTE-CARLO (RL "from scratch", NumPy) -----
# État : (temps, inventaire, popularité, rythme de vente, prix vs référence, Fan Cost Index)
#        discrétisé sur STATE_BINS (6 480 états). Popularité + FCI situent le
#        scénario (niche vs superstar, club vs stade) ; rythme + prix/référence
#        servent à ajuster en cours de vente.
# Action : palier de prix, RÉVISÉ tous les `steps_per_decision` pas de marché
#          (cadence grossière -> horizon de décision ~12 -> crédit temporel gérable).
# Récompense : marge de contribution du bloc, normalisée par épisode
#              = (prix - marginal_cost + ancillary_capture*FCI)*ventes / (1.5*demande convertible)
# Mise à jour : Monte-Carlo on-policy (Sutton & Barto ch. 5). Après CHAQUE épisode
#   complet, on remonte la trajectoire : G_t = r_t + gamma*G_{t+1}, puis
#   Q(s,a) <- Q(s,a) + alpha*(G_t - Q(s,a)).  Pas de bootstrapping -> cible non
#   biaisée (la cible TD est trop instable sur cet environnement à forte variance).
#
# Checkpoint tous les 2000 épisodes : RESUME = True pour reprendre après coupure.

_reload_project()                                  # resync chaîne complète (robuste au kernel non redémarré)
from gym_wrapper import STATE_BINS, TicketEnv, discretize_observation
env = TicketEnv(horizon=HORIZON)                    # env FRAIS

_info0 = env.reset(seed=0)[1]                       # garde-fou (cf. cellule des imports)
if "ca_total" not in _info0 or not hasattr(env, "steps_per_decision"):
    raise RuntimeError("Environnement périmé — fais 'Kernel > Restart Kernel' puis Run All.")

from registry import checkpoint_path, model_path, write_meta

RESUME = False
CKPT = str(checkpoint_path("q_table"))
STATE_FILE = CKPT + ".state.json"
episodes = 60000                                # grille 6 480 états -> ~50-70k épisodes

n_actions = env.action_space.n
alpha = 0.05          # pas d'apprentissage Monte-Carlo (à alpha constant)
gamma = 0.97          # facteur d'actualisation
eps_start, eps_min, eps_frac = 1.0, 0.06, 0.75  # ε linéaire, minimum atteint à 75 %
rng = np.random.default_rng(0)
q_shape = (*STATE_BINS, n_actions)

def epsilon(ep):
    return max(eps_min, eps_start + (eps_min - eps_start) * ep / (eps_frac * episodes))

if RESUME and os.path.exists(CKPT) and os.path.exists(STATE_FILE):
    q_table = np.load(CKPT)
    if q_table.shape != q_shape:
        raise ValueError(f"Checkpoint de forme {q_table.shape}, attendu {q_shape}. "
                         f"Supprime {CKPT} (+ .state.json) et relance avec RESUME = False.")
    st = json.load(open(STATE_FILE, encoding="utf-8"))
    start_episode, episode_returns = st["episode"], st["returns"]
    print(f"Reprise à l'épisode {start_episode}")
else:
    q_table = np.zeros(q_shape)
    start_episode, episode_returns = 0, []
    print(f"Entraînement Monte-Carlo depuis zéro sur {episodes} épisodes "
          f"({HORIZON // env.steps_per_decision} décisions/épisode)...")

os.makedirs(os.path.dirname(CKPT), exist_ok=True)
for episode in range(start_episode, episodes):
    eps = epsilon(episode)
    obs, _ = env.reset(seed=episode)          # une graine par épisode -> DR reproductible
    state = discretize_observation(obs)
    traj = []
    done = False
    while not done:
        action = int(rng.integers(n_actions)) if rng.random() < eps else int(np.argmax(q_table[state]))
        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        traj.append((state, action, reward))
        state = discretize_observation(next_obs)

    # remontée Monte-Carlo (retour à rebours, sans bootstrapping)
    g = 0.0
    for st_, a_, r_ in reversed(traj):
        g = r_ + gamma * g
        q_table[st_][a_] += alpha * (g - q_table[st_][a_])
    episode_returns.append(float(sum(r_ for _, _, r_ in traj)))

    if (episode + 1) % 2000 == 0:
        np.save(CKPT, q_table)
        json.dump({"episode": episode + 1, "returns": episode_returns},
                  open(STATE_FILE, "w", encoding="utf-8"))
        print(f"  épisode {episode + 1:>6d}/{episodes}  eps={eps:.3f}  "
              f"retour_moyen={np.mean(episode_returns[-500:]):.2f}")

visited = int(np.count_nonzero(q_table.any(axis=-1)))
print(f"Terminé. États visités : {visited}/{int(np.prod(STATE_BINS))}")

w = 400
smooth = np.convolve(episode_returns, np.ones(w) / w, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(smooth)
plt.title("Q-table (Monte-Carlo) — retour par épisode (moyenne glissante 400)")
plt.xlabel("Épisode"); plt.ylabel("Retour (récompense normalisée cumulée)")
plt.grid(alpha=0.3)
plt.show()

In [19]:
# --- Sauvegarde horodatée --------------------------------------------------
final = model_path("q_table")                       # q_table_AAAAMMJJ-HHMMSS.npy
np.save(final, q_table)
write_meta(final, kind="q_table", method="monte_carlo", episodes=len(episode_returns),
           horizon=HORIZON, steps_per_decision=env.steps_per_decision,
           seed=0, alpha=alpha, gamma=gamma, state_bins=list(STATE_BINS),
           states_visited=visited, states_total=int(np.prod(STATE_BINS)))

for f in (CKPT, STATE_FILE):                        # checkpoint terminé -> nettoyage
    if os.path.exists(f):
        os.remove(f)

print(f"Modèle sauvegardé : {final.name}  ({visited}/{int(np.prod(STATE_BINS))} états visités)")
print("Les agents chargent cette version (la plus récente) par défaut : QTableAgent.load()")

Modèle sauvegardé : q_table_20260830-164143.npy  (3899/6480 états visités)
Les agents chargent cette version (la plus récente) par défaut : QTableAgent.load()


In [20]:
# resync la chaîne complète + les agents de pricing (au cas où le kernel a une version périmée)
_reload_project()                                   # refdata -> agents -> market -> gym_wrapper
for _name in ("registry", "agent_fixed", "agent_qtable", "evaluate"):
    importlib.reload(importlib.import_module(_name))

# --- Contrôle rapide : la politique apprise bat-elle le prix fixe ? ----------
from agent_fixed import FixedPriceAgent
from agent_qtable import QTableAgent
from evaluate import compare_agents, summarize

agents = [
    FixedPriceAgent.from_price(70),      # prix fixe "réaliste" (instinct : remplir la salle)
    FixedPriceAgent.from_price(90),      # prix fixe optimum (inconnaissable a priori)
    QTableAgent(q_table),                # politique fraîchement entraînée
]
agents[0].name = "fixed"
agents[1].name = "fixed_90"

results = compare_agents(agents, lambda: TicketEnv(horizon=HORIZON), n_episodes=300)
summarize(results)

,revenue_mean,ca_mean,revenue_std,fill_rate_mean,tickets_sold_mean,n,revenue_vs_fixed_pct,ca_vs_fixed_pct
agent,,,,,,,,
fixed_90,188577.415833,220664.365833,205739.408927,0.736360,2139.130000,300,14.131283,9.446998
fixed,165228.508333,201617.558333,169216.603881,0.851139,2425.936667,300,0.000000,0.000000
qtable,123614.571667,161204.321667,138576.861371,0.900281,2505.983333,300,-25.185688,-20.044503
